In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders.image import UnstructuredImageLoader
from langchain_unstructured import UnstructuredLoader


In [5]:
load_dotenv()

#create llm object
llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash")

loader = UnstructuredLoader("sample_unstructured_test.pdf")
documents = loader.load()

INFO: pikepdf C++ to Python logger bridge initialized


In [6]:
#split the text using recursive approch
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)
chunks = text_splitter.split_documents(documents)

In [7]:
#Embedding model 
embedding=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

#------------------generate embeddings----------------------
embeddings=embedding.embed_documents([text.page_content for text in chunks])   

#create database from chunks and embedding model                                                           
db=FAISS.from_documents(chunks,embedding)                                       

INFO: Loading faiss with AVX2 support.
INFO: Successfully loaded faiss with AVX2 support.
INFO: Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


In [8]:
db_save_path = "db/faiss_index_gemini"
db.save_local(db_save_path)
loaded_db=FAISS.load_local(db_save_path,embedding,allow_dangerous_deserialization=True)
question="What is the backend for the project?"
similler_data=loaded_db.similarity_search(question,k=2)

: 

In [ ]:

# prompt template and generation                                                                
prompt_template = """
Use the following context to answer the question:

{context}

Question: {question}

Answer:
"""

prompt=PromptTemplate.from_template(prompt_template)
formatted_prompt=prompt.invoke({
    "context":similler_data[0].page_content,
    "question":question
})
# print(formatted_prompt)
final_response=llm.invoke(formatted_prompt)
print(final_response.content)

In [ ]:
# from langchain_unstructured import UnstructuredLoader

# loader = UnstructuredLoader("sample_unstructured_test.pdf")
# documents = loader.load()
# for element in documents:
#     print(element.page_content)
#     print(element.metadata.get("category"))

from langchain_community.document_loaders.image import UnstructuredImageLoader

loader = UnstructuredImageLoader("sc.jpg")
image_docs = loader.load()